# Teste do Modelo 4: Google TimesFM

In [1]:
import pandas as pd
import torch
import numpy as np
import os

# --- 1. Configurações ---
DATA_DIR = "../../data"
HORIZONTE_PREVISAO = 14
CONTEXT_LENGTH = 96 # O contexto deve ser um múltiplo de 32

# --- 2. Carregar Dados ---
print("Carregando dados...")
hist_path = os.path.join(DATA_DIR, "hist.parquet")
df_hist = pd.read_parquet(hist_path)
print("Dados históricos carregados.")

# --- 3. Definir dispositivo ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsando dispositivo: {device}")

Carregando dados...
Dados históricos carregados.

Usando dispositivo: cpu


In [2]:
# ==============================================================================
# TESTE 4: Google TimesFM (API Atualizada)
# ==============================================================================
# A API foi atualizada para carregar o modelo diretamente do Hugging Face Hub.
# ==============================================================================

try:
    import timesfm
    
    print("\n--- Testando Google TimesFM ---")

    # Garantir que temos dados suficientes
    if len(df_hist) < CONTEXT_LENGTH:
        raise ValueError(f"TimesFM precisa de pelo menos {CONTEXT_LENGTH} pontos de dados.")

    # 1. Carregar o modelo a partir do Hugging Face Hub
    torch.set_float32_matmul_precision("high")
    model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
        "google/timesfm-2.5-200m-pytorch",
        device_map=device # Correção: Atribuir dispositivo no carregamento
    )
    # model.to(device) # Linha com bug removida

    # 2. Configurar a previsão
    forecast_config = timesfm.ForecastConfig(
        max_context=CONTEXT_LENGTH,
        max_horizon=HORIZONTE_PREVISAO,
        normalize_inputs=True,
        use_continuous_quantile_head=False, # Para previsão de ponto
    )
    model.compile(forecast_config)

    # 3. Preparar dados de entrada
    context_data = df_hist['target'].values[-CONTEXT_LENGTH:]
    
    # 4. Rodar a previsão
    print(f"Rodando previsão para {HORIZONTE_PREVISAO} passos...")
    point_forecast, _ = model.forecast(
        horizon=HORIZONTE_PREVISAO,
        inputs=[context_data],
    )

    # Extrair resultados
    forecast_values = point_forecast[0]

    print("Previsão (primeiros 5 valores):", forecast_values.round(2)[:5])
    print("Teste do TimesFM concluído.\n")

except ImportError:
    print("TimesFM não instalado. Pulando teste.")
except Exception as e:
    print(f"Erro ao rodar TimesFM: {e}")


--- Testando Google TimesFM ---


/app/time_series_models/notebooks/04_timesfm/.venv/lib/python3.12/site-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

Downloaded.


Rodando previsão para 14 passos...


Previsão (primeiros 5 valores): [114.16 106.43  94.88  87.34  90.97]
Teste do TimesFM concluído.

